# Basic Machine Learning Model - Philipp

In [5]:
# Relevant imports

from datetime import datetime
import pandas as pd
import sys
from pathlib import Path
import numpy as np
import re
from time import perf_counter
from __future__ import annotations


from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score



repo_root = Path.cwd().parent.parent
notebook_dir = Path.cwd().parent

for p in [repo_root, notebook_dir]:
    if str(p) not in sys.path:
        sys.path.append(str(p))

print(f"Repo root: {repo_root}")


from sintef_project.kaggle_metric import score
from sintef_project.submission import create_submission

from data_loaders import Training_set

Repo root: c:\Users\phili\Documents\MMLP_project


## Helper Functions for Data Handling

In [6]:
"""Long-format models without autoregressive commitment lags."""

TARGET_PATTERN = re.compile(r"^result_committed_(?P<unit>.+)_t(?P<hour>\d+)$")
T = 168


def wide_to_long_X(X_wide: pd.DataFrame) -> pd.DataFrame:
    """Convert one wide row per run into one row per run and horizon hour."""
    hourly_bases = sorted({
        column.rsplit("_t", 1)[0]
        for column in X_wide.columns
        if "_t" in column
        and column.rsplit("_t", 1)[1].isdigit()
        and not column.startswith(("price_hour_sin_", "price_hour_cos_"))
    })
    daily_bases = sorted({
        column.rsplit("_day", 1)[0]
        for column in X_wide.columns
        if "_day" in column and column.rsplit("_day", 1)[1].isdigit()
    })
    static_columns = [
        column for column in X_wide.columns
        if column not in {"Run No", "starttime"}
        and "_t" not in column
        and "_day" not in column
    ]

    hour_of_day = np.tile(np.arange(T, dtype=np.int16) % 24, len(X_wide))
    columns = [
        np.sin(2 * np.pi * hour_of_day / 24),
        np.cos(2 * np.pi * hour_of_day / 24),
    ]
    for base in hourly_bases:
        columns.append(
            X_wide[[f"{base}_t{hour}" for hour in range(T)]]
            .to_numpy(dtype=np.float32)
            .ravel()
        )
    for base in daily_bases:
        daily_values = X_wide[
            [f"{base}_day{day}" for day in range(7)]
        ].to_numpy(dtype=np.float32)
        columns.append(np.repeat(daily_values, 24, axis=1).ravel())
    for column in static_columns:
        columns.append(np.repeat(X_wide[column].to_numpy(dtype=np.float32), T))

    column_names = ["hour_sin", "hour_cos"] + hourly_bases + daily_bases + static_columns
    return pd.DataFrame(np.column_stack(columns), columns=column_names)


def get_unit_names(y_columns: pd.Index) -> list[str]:
    """Extract unique generator names from wide target columns."""
    units = {
        match.group("unit")
        for column in y_columns
        if (match := TARGET_PATTERN.match(column))
    }
    if not units:
        raise ValueError("No commitment target columns were found.")
    return sorted(units)


def _positive_class_probability(model, features: pd.DataFrame) -> np.ndarray:
    probabilities = model.predict_proba(features)
    if len(model.classes_) == 1:
        return probabilities[:, 0] if model.classes_[0] == 1 else probabilities[:, 0] * 0
    return probabilities[:, 1]


def fit_long_models_with_progress(estimator, X_train_wide, y_train_wide):
    """Fit one model per generator using the shared long-format features."""
    X_long = wide_to_long_X(X_train_wide)
    units = get_unit_names(y_train_wide.columns)
    models = {}
    training_start = perf_counter()

    for unit_number, unit in enumerate(units, start=1):
        target_start = perf_counter()
        target_columns = [f"result_committed_{unit}_t{hour}" for hour in range(T)]
        missing = [column for column in target_columns if column not in y_train_wide]
        if missing:
            raise ValueError(f"Missing target columns for {unit}: {missing[:3]}")

        X_unit = X_long
        target_values = y_train_wide[target_columns].to_numpy(dtype=np.uint8).ravel()
        if np.unique(target_values).size == 1:
            model = DummyClassifier(strategy="constant", constant=target_values[0])
        else:
            model = clone(estimator)
        model.fit(X_unit, target_values)
        models[unit] = model

        elapsed = (perf_counter() - training_start) / 60
        last_model = (perf_counter() - target_start) / 60
        eta = (elapsed / unit_number) * (len(units) - unit_number)
        print(
            f"[{unit_number}/{len(units)}] {unit} finished | "
            f"last model: {last_model:.2f} min | elapsed: {elapsed:.2f} min | "
            f"ETA: {eta:.2f} min",
            flush=True,
        )

    return models


def positive_class_probabilities(fitted_models, X_wide, y_columns):
    """Predict each generator and reconstruct the original wide target shape."""
    X_long = wide_to_long_X(X_wide)
    n_rows = len(X_wide)
    predictions = {}

    for unit, model in fitted_models.items():
        probabilities = _positive_class_probability(model, X_long)

        probability_matrix = probabilities.reshape(n_rows, T)
        for hour in range(T):
            predictions[f"result_committed_{unit}_t{hour}"] = probability_matrix[:, hour]

    result = pd.DataFrame(predictions, index=X_wide.index)
    missing = [column for column in y_columns if column not in result]
    if missing:
        raise ValueError(f"Missing prediction columns: {missing[:3]}")
    return result.loc[:, list(y_columns)]


def evaluate_generator_performance(y_true, predictions):
    """Return 2022 performance metrics for every generator."""
    if not y_true.index.equals(predictions.index):
        raise ValueError("y_true and predictions must have identical indexes.")

    rows = []
    for unit in get_unit_names(y_true.columns):
        target_columns = [f"result_committed_{unit}_t{hour}" for hour in range(T)]
        missing_true = [column for column in target_columns if column not in y_true]
        missing_predictions = [column for column in target_columns if column not in predictions]
        if missing_true or missing_predictions:
            raise ValueError(f"Missing target columns for {unit}.")

        actual = y_true[target_columns].to_numpy(dtype=np.uint8).ravel()
        predicted = predictions[target_columns].to_numpy(dtype=float).ravel()
        predicted_labels = (predicted >= 0.5).astype(np.uint8)
        roc_auc = (
            roc_auc_score(actual, predicted)
            if np.unique(actual).size == 2
            else np.nan
        )
        rows.append({
            "generator": unit,
            "roc_auc": roc_auc,
            "accuracy_at_0.5": accuracy_score(actual, predicted_labels),
            "actual_commitment_rate": actual.mean(),
            "predicted_commitment_rate_at_0.5": predicted_labels.mean(),
            "observations": len(actual),
        })

    return pd.DataFrame(rows).sort_values("roc_auc", ascending=False, na_position="last").reset_index(drop=True)


def show_training_week_example(X, y, week_start=None, generator=None, rows_to_print=24):
    """Show the model input and target for one training week and generator."""
    if "starttime" not in X:
        raise ValueError("X must contain a 'starttime' column.")
    if len(X) == 0 or len(y) == 0:
        raise ValueError("X and y must contain training data.")
    if not X.index.equals(y.index):
        raise ValueError("X and y must have identical indexes.")

    units = get_unit_names(y.columns)
    if generator is None:
        generator = units[0]
    if generator not in units:
        raise ValueError(f"Unknown generator {generator!r}. Choose one of: {units}")

    available_starts = pd.to_datetime(X["starttime"])
    if week_start is None:
        selected_start = available_starts.iloc[0]
    else:
        requested_start = pd.Timestamp(week_start)
        matching_starts = available_starts[available_starts >= requested_start]
        if matching_starts.empty:
            raise ValueError("No training week starts on or after week_start.")
        selected_start = matching_starts.iloc[0]

    selected_index = available_starts[available_starts == selected_start].index
    X_week = X.loc[selected_index]
    y_week = y.loc[selected_index]
    model_input = wide_to_long_X(X_week)
    target_columns = [f"result_committed_{generator}_t{hour}" for hour in range(T)]
    target_values = y_week[target_columns].to_numpy(dtype=np.uint8).ravel()

    example = model_input.copy()
    example.insert(0, "hour_in_week", np.arange(len(example)))
    example.insert(1, "starttime", selected_start)
    example["generator"] = generator
    example["target_commitment"] = target_values

    print(f"Training-week example: {selected_start} to {selected_start + pd.Timedelta(hours=T - 1)}")
    print(f"Generator: {generator}")
    print(f"Model input shape: {model_input.shape}")
    print(f"Target column: result_committed_{generator}_t0 ... result_committed_{generator}_t167")
    print(f"Showing the first {min(rows_to_print, len(example))} of {len(example)} hourly rows:")
    print(example.head(rows_to_print).to_string(index=False))
    return example


## Data Preprocessing

In [7]:
# Topology Information:

## Generator Locations:
gen_locations = {
    "Byrte": ["Byrte_G1"],
    "Haukeli": ["Haukeli_G1"],
    "Hogga": ["Hogga_G1"],
    "Kjela": ["Kjela_G1"],
    "Lio": ["Lio_G1"],
    "Songa": ["Songa_G1"],
    "Tokke": ["Tokke_G1", "Tokke_G2", "Tokke_G3", "Tokke_G4"],
    "Vesle Kjela": ["Vesle Kjela_G1"],
    "Vinje": ["Vinje_G1", "Vinje_G2", "Vinje_G3"],
}



In [8]:
## Define the training set

class Training_base(Training_set):
    
    def raw_preprocessing(self):
        raw = self.load_copies()
        
        # 1. Extract raw dataframes for price, volume, and water value
        price_df = raw["day_ahead_price"].copy()
        volume_df = raw["volume"].copy()
        water_value_df = raw["water_value"].copy()

        
        
        # 2. Engineer Feature A: Daily Relative Price (Normalized by daily mean)
        price_df["date_only"] = price_df.index.normalize()
        daily_mean_price = price_df.groupby("date_only")["day_ahead_price"].transform("mean")
        price_df["price_relative_to_daily_mean"] = price_df["day_ahead_price"] - daily_mean_price
        price_df.drop(columns=["date_only"], inplace=True)
        
        # 3. Engineer Feature B: Economic Spreads (Price minus Water Value per reservoir)
        # Since water_value is daily and price_df is hourly, we forward-fill water values to hourly index 
        # or compute it dynamically during windowing. A clean way in preprocessing is to merge/align them:
        wv_hourly = water_value_df.reindex(price_df.index, method="ffill")
        
        spread_frames = {}
        for res_col in water_value_df.columns:
            # Create a spread column for each reservoir: Price minus Water Value
            spread_col_name = f"spread_{res_col}"
            spread_frames[spread_col_name] = price_df["day_ahead_price"] - wv_hourly[res_col]
            
        spread_df = pd.DataFrame(spread_frames, index=price_df.index)
        
        # Combine everything back into the dictionary that prepare_data expects
        processed_sources = {
            "volume": volume_df,
            "price_relative_to_daily_mean": price_df[["price_relative_to_daily_mean"]],
            "spread": spread_df,
        }
        
        return processed_sources


class Training_advanced(Training_set):
    
    def raw_preprocessing(self):
        raw = self.load_copies()
        
        # 1. Extract raw dataframes for price, volume, and water value
        price_df = raw["day_ahead_price"].copy()
        volume_df = raw["volume"].copy()
        water_value_df = raw["water_value"].copy()

        # Hourly cyclical time features directly in preprocessing
        hour = price_df.index.hour
        price_df["hour"] = hour
        price_df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
        price_df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
        
        # 2. Engineer Feature A: Daily Relative Price (Normalized by daily mean)
        price_df["date_only"] = price_df.index.normalize()
        daily_mean_price = price_df.groupby("date_only")["day_ahead_price"].transform("mean")
        price_df["price_relative_to_daily_mean"] = price_df["day_ahead_price"] - daily_mean_price
        price_df.drop(columns=["date_only"], inplace=True)

        # 3. Engineer Feature B: Economic Spreads (Price minus Water Value per reservoir)
        wv_hourly = water_value_df.reindex(price_df.index, method="ffill")

        # 4. For each reservoir compute a spread colum: Price minus Water Value
        spread_frames = {}
        for res_col in water_value_df.columns:
            # Create a spread column for each reservoir: Price minus Water Value
            spread_col_name = f"spread_{res_col}"
            spread_frames[spread_col_name] = price_df["day_ahead_price"] - wv_hourly[res_col]
            
        spread_df = pd.DataFrame(spread_frames, index=price_df.index)

        # 5. Engineer Feature C: Water Value Difference (Forward Difference) - Is the water value increasing or decreasing?
        wv_diff_forward = water_value_df.shift(-1) - water_value_df
        wv_diff_forward.columns = [f"{col}_wv_diff_forward" for col in wv_diff_forward.columns]
        
        wv_hourly = wv_diff_forward.reindex(price_df.index, method="ffill")

        # Day feature day of week
        day_features_df = pd.DataFrame(index=water_value_df.index)
        day_features_df["dayofweek"] = day_features_df.index.dayofweek
        day_features_df["is_weekend"] = (day_features_df.index.dayofweek >= 5).astype(int)


        # Combine everything back into the dictionary that prepare_data expects
        processed_sources = {
            "volume": volume_df,
            "day_features": day_features_df,
            "price": price_df,
            "spread": spread_df,
            "water_value": wv_diff_forward
        }

        # Also determine for each time step

        return processed_sources

training_set = Training_advanced()

In [9]:
## Defining the training and test periods
start_date = datetime(2015, 1, 1)
end_date = datetime(2022, 12, 31)

hour_memory = 24 # For basic model no historical data is used, so hour_memory is set to 0.

X, y = training_set.prepare_data(
    start_date=start_date,
    end_date=end_date,
    hour_memory=hour_memory,
)

training_week_example = show_training_week_example(
    X,
    y,
    week_start="2015-01-02",
    generator=get_unit_names(y.columns)[0],
)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("X index:", X.index)
print("y index:", y.index)

print("Same index:", X.index.equals(y.index))
print("Same number of rows:", len(X) == len(y))

# Split the data into training, validation, and test sets based on the starttime column
train_val_mask = X['starttime'] < datetime(2022, 1, 1)
test_mask = X['starttime'] >= datetime(2022, 1, 1)

X_train_val = X.loc[train_val_mask].copy()
y_train_val = y.loc[train_val_mask].copy()

training_set.export_X_and_Y_summary(X_train_val, y_train_val, name="base_model_train_val")

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()


folds = [
    {
        'train_end': '2018-01-01',
        'val_start': '2018-01-01',
        'val_end': '2019-01-01'
    },
    {
        'train_end': '2019-01-01',
        'val_start': '2019-01-01',
        'val_end': '2020-01-01'
    },
    {
        'train_end': '2020-01-01',
        'val_start': '2020-01-01',
        'val_end': '2021-01-01'
    },
    {
        'train_end': '2021-01-01',
        'val_start': '2021-01-01',
        'val_end': '2022-01-01'
    }
]

train_folds = []

for fold in folds:

    train_mask = (
        X['starttime'] < pd.Timestamp(fold['train_end'])
    )

    val_mask = (
        (X['starttime'] >= pd.Timestamp(fold['val_start'])) &
        (X['starttime'] < pd.Timestamp(fold['val_end']))
    )

    X_train = X.loc[train_mask]
    y_train = y.loc[train_mask]

    X_val = X.loc[val_mask]
    y_val = y.loc[val_mask]

    train_folds.append((X_train, y_train, X_val, y_val))

    print(
        f"Train: {X_train['starttime'].min()} "
        f"to {X_train['starttime'].max()}"
    )

    print(
        f"Validation: {X_val['starttime'].min()} "
        f"to {X_val['starttime'].max()}"
    )

X shape: (2921, 4516)
y shape: (2921, 2352)
X index: RangeIndex(start=0, stop=2921, step=1)
y index: RangeIndex(start=0, stop=2921, step=1)
Same index: True
Same number of rows: True
Train: 2015-01-02 00:00:00 to 2017-12-31 00:00:00
Validation: 2018-01-01 00:00:00 to 2018-12-31 00:00:00
Train: 2015-01-02 00:00:00 to 2018-12-31 00:00:00
Validation: 2019-01-01 00:00:00 to 2019-12-31 00:00:00
Train: 2015-01-02 00:00:00 to 2019-12-31 00:00:00
Validation: 2020-01-01 00:00:00 to 2020-12-31 00:00:00
Train: 2015-01-02 00:00:00 to 2020-12-31 00:00:00
Validation: 2021-01-01 00:00:00 to 2021-12-31 00:00:00


In [ ]:

# Define the base classifier (n_jobs=-1 is crucial here to use all CPU cores!)
estimator = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    n_jobs=-1,  
    random_state=42,
)

# Optional time-based cross-validation
avg_score = None
bool_train_folds = True
if bool_train_folds:
    validation_scores = []
    for i, (X_train_full, y_train, X_val_full, y_val) in enumerate(train_folds):
        print(f"\n--- Starting Training for Fold {i+1} ---")
        time_start = datetime.now()
        print(f"Training period: {X_train_full['starttime'].min()} to {X_train_full['starttime'].max()}")
        print(f"Validation period: {X_val_full['starttime'].min()} to {X_val_full['starttime'].max()}")
        
        # Fit one long-format model per generator
        model = fit_long_models_with_progress(
            estimator=estimator,
            X_train_wide=X_train_full,
            y_train_wide=y_train,
        )
        
        # Generate predictions
        val_predictions = positive_class_probabilities(model, X_val_full, y_train.columns)
        
        # Calculate and store the score
        fold_score = score(y_val, val_predictions)
        validation_scores.append(fold_score)
        print(f"-> Fold {i+1} Validation micro ROC-AUC: {fold_score:.4f}")
        print(f"-> Fold {i+1} Training Time: {datetime.now() - time_start}")
        
    # Output the average performance across all folds
    avg_score = sum(validation_scores) / len(validation_scores)
    print(f"\n=========================================")
    print(f"Average CV ROC-AUC: {avg_score:.4f}")
    print(f"=========================================\n")


# Final training and testing on the held-out 2022 data
print("--- Final Model on 2022 Test Data ---")
test_mask = X['starttime'] >= pd.Timestamp("2022-01-01")
train_val_mask = X['starttime'] < pd.Timestamp("2022-01-01")

# Use all knowledge from 2015-2021 for the final model
y_train_val = y.loc[train_val_mask]

y_test = y.loc[test_mask]

final_model = fit_long_models_with_progress(
    estimator=estimator,
    X_train_wide=X.loc[train_val_mask],
    y_train_wide=y_train_val
)

test_predictions = positive_class_probabilities(final_model, X.loc[test_mask], y_train_val.columns,
    )
test_score = score(y_test, test_predictions)
print(f"-> Final Test micro ROC-AUC (2022): {test_score:.4f}")

generator_performance = evaluate_generator_performance(y_test, test_predictions)
print("\n2022 performance per generator:")
print(generator_performance.to_string(index=False, float_format=lambda value: f"{value:.4f}"))


### Add the average ROC-AUC across folds and the final test score to a log file to keep track of model performance over time
additional_context = "Model using XGBoost with engineered features: daily relative price, economic spreads (price - water_values), water value forward differences, and cyclical time features. " \
"Uses a memory of 24 hours without autoregressive commitment lags."


log_file_path = repo_root / "model_performance_log.txt"
with open(log_file_path, "a") as log_file:
    log_file.write(f"Date: {datetime.now()}\n")
    log_file.write(f"Context: {additional_context}\n")
    if avg_score is not None:
        log_file.write(f"Average CV ROC-AUC: {avg_score:.4f}\n")
    log_file.write(f"Final Test ROC-AUC (2022): {test_score:.4f}\n")
    log_file.write("Per-generator 2022 performance:\n")
    for _, row in generator_performance.iterrows():
        log_file.write(
            f"{row['generator']}: ROC-AUC={row['roc_auc']:.4f}, "
            f"accuracy@0.5={row['accuracy_at_0.5']:.4f}, "
            f"actual_rate={row['actual_commitment_rate']:.4f}, "
            f"predicted_rate@0.5={row['predicted_commitment_rate_at_0.5']:.4f}, "
            f"observations={int(row['observations'])}\n"
        )
    log_file.write("-" * 40 + "\n")


--- Starting Training for Fold 1 ---
Training period: 2015-01-02 00:00:00 to 2017-12-31 00:00:00
Validation period: 2018-01-01 00:00:00 to 2018-12-31 00:00:00
[1/14] Byrte_G1 finished | last model: 0.04 min | elapsed: 0.04 min | ETA: 0.49 min
[2/14] Haukeli_G1 finished | last model: 0.04 min | elapsed: 0.08 min | ETA: 0.46 min
[3/14] Hogga_G1 finished | last model: 0.04 min | elapsed: 0.12 min | ETA: 0.42 min
[4/14] Kjela_G1 finished | last model: 0.04 min | elapsed: 0.15 min | ETA: 0.38 min
[5/14] Lio_G1 finished | last model: 0.04 min | elapsed: 0.19 min | ETA: 0.34 min
[6/14] Songa_G1 finished | last model: 0.04 min | elapsed: 0.23 min | ETA: 0.30 min
[7/14] Tokke_G1 finished | last model: 0.04 min | elapsed: 0.27 min | ETA: 0.27 min
[8/14] Tokke_G2 finished | last model: 0.04 min | elapsed: 0.31 min | ETA: 0.23 min
[9/14] Tokke_G3 finished | last model: 0.04 min | elapsed: 0.35 min | ETA: 0.20 min
[10/14] Tokke_G4 finished | last model: 0.04 min | elapsed: 0.39 min | ETA: 0.16 min

## Creating the submission format

In [ ]:
## Create Kaggle Submission data
X_kaggle, _ = training_set.prepare_data(
    start_date=datetime(2023, 1, 1),
    end_date=datetime(2024, 12, 25),
    hour_memory=hour_memory,
    prediction_mapping_path=repo_root / "sintef_project" / "prediction_mapping.csv",
)

training_set.export_X_and_Y_summary(X_kaggle, None, name="submission_input")

print(
    "Kaggle date range:",
    X_kaggle["starttime"].min(),
    "to",
    X_kaggle["starttime"].max(),
)

kaggle_predictions = positive_class_probabilities(
    fitted_models=final_model,
    X_wide=X_kaggle,
    y_columns=y_train_val.columns,
)


kaggle_predictions.insert(
    0,
    "Run No",
    X_kaggle["Run No"].to_numpy(),
)



# Create Kaggle submission file

submission_path = create_submission(
    predictions=kaggle_predictions,
    identifier="basic_model_without_2022",
)

print(f"Kaggle submissions saved here: {submission_path}")

Kaggle date range: 2023-01-01 00:00:00 to 2024-12-25 00:00:00
Kaggle submissions saved here: output\basic_model_without_2022.csv
